In [ ]:
from c4_otimes_c4 import pianni
from CmCn_entangled.convex_combination import convex_combination
import numpy as np
from SDP_utils.sdp import SDP
from toqito.states import max_mixed
from functools import reduce
from toqito.matrix_ops import partial_trace
tol = 1e-8

In [2]:
state035: np.ndarray = list(convex_combination(pianni(), max_mixed(16), 0.35, 0.4, 0.5))[0][1] #type:ignore
print(state035)

[[ 0.05520833+0.j  0.        +0.j  0.        +0.j -0.01458333+0.j
   0.        +0.j  0.        +0.j  0.        +0.j  0.        +0.j
   0.        +0.j  0.        +0.j  0.        +0.j  0.        +0.j
  -0.01458333+0.j  0.        +0.j  0.        +0.j  0.01458333+0.j]
 [ 0.        +0.j  0.084375  +0.j -0.01458333+0.j  0.        +0.j
   0.        +0.j  0.        +0.j  0.        +0.j  0.        +0.j
   0.        +0.j  0.        +0.j  0.        +0.j  0.        +0.j
   0.        +0.j -0.01458333+0.j -0.01458333+0.j  0.        +0.j]
 [ 0.        +0.j -0.01458333+0.j  0.084375  +0.j  0.        +0.j
   0.        +0.j  0.        +0.j  0.        +0.j  0.        +0.j
   0.        +0.j  0.        +0.j  0.        +0.j  0.        +0.j
   0.        +0.j -0.01458333+0.j -0.01458333+0.j  0.        +0.j]
 [-0.01458333+0.j  0.        +0.j  0.        +0.j  0.05520833+0.j
   0.        +0.j  0.        +0.j  0.        +0.j  0.        +0.j
   0.        +0.j  0.        +0.j  0.        +0.j  0.        +0.j
   0.01

In [3]:
# SDP([1,1], state035, 4,4)

In [4]:
eigenvalues, eigenvectors = np.linalg.eigh(state035)
rho_reconstructed = sum(
    eigenvalues[i] * np.outer(eigenvectors[:, i], eigenvectors[:, i].conj())
    for i in range(len(eigenvalues))
)
print(np.allclose(state035, rho_reconstructed))

True


In [5]:
d,k = 4,3
feasible_val = np.zeros((d**2,d**2), dtype=np.complex128)
for a_i, psi_i in zip(eigenvalues, eigenvectors.T):
    feasible_val += a_i * np.outer(psi_i, psi_i.conj())

print(np.allclose(state035, feasible_val))
print(state035.dtype)

True
complex128


In [6]:
d,k = 4,3
omega_1k: np.ndarray = np.zeros((d**(k*2),d**(k*2)), dtype=np.complex128)
for a_i, psi_i in zip(eigenvalues, eigenvectors.T):
    psi_i_kron = reduce(np.kron, [psi_i] * k)
    omega_1k += a_i * np.outer(psi_i_kron, psi_i_kron.conj())
recon_rho = partial_trace(omega_1k, list(range(1, k)), [d**2] * k)

print(np.allclose(state035, recon_rho)) # type: ignore


True


In [7]:
from SDP_utils.SDP_matrices import V_builder, W_l_builder, alpha_dag_j_builder
V = V_builder(k, d**2)
omega_sym = V @ omega_1k @ V.conj().T
print(np.trace(omega_sym))
sym_eigen = np.linalg.eigvalsh(omega_sym)
print(eigenvalues)

(1.0000000000000004+0j)
[0.040625   0.040625   0.040625   0.040625   0.040625   0.040625
 0.040625   0.040625   0.040625   0.040625   0.09895833 0.09895833
 0.09895833 0.09895833 0.09895833 0.09895833]


In [8]:
pi = pianni()
print(np.linalg.eigvalsh(pi))

[-2.54630587e-17 -1.39756495e-17 -1.24786085e-17 -1.10803585e-17
 -5.71007771e-18 -2.58819295e-18  2.66312700e-18  1.05004243e-17
  2.08166817e-17  2.78116307e-17  1.66666667e-01  1.66666667e-01
  1.66666667e-01  1.66666667e-01  1.66666667e-01  1.66666667e-01]
